In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-ridho-model'  # ckpt = 4000
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-25-norand'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # min_ankle_height 弊害
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
exp_name = 'bp000-walking'
# exp_name = 'friction-walking-fractal-norand'
ckpt = 2000

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
env_cfg["episode_length_s"] = 60.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
env_cfg['dt'] = 0.01
env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2500.0,
 'kd': 700.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 60.0,
 'resampling_time_s': 4.0,
 'action_scale': 0.25,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0]}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-1.9162e-01,  8.1881e-01, -8.9760e-02,  1.4423e+00, -1.5019e+00,
          1.0640e+00,  1.4185e+00,  1.4308e-01,  1.0860e-03,  9.4490e-01,
         -1.3915e-01,  4.5646e-01]], device='cuda:0')
Scaled actions :  tensor([[-1.9162e-01,  8.1881e-01, -8.9760e-02,  1.4423e+00, -1.5019e+00,
          1.0640e+00,  1.4185e+00,  1.4308e-01,  1.0860e-03,  9.4490e-01,
         -1.3915e-01,  4.5646e-01]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-1.7201e-06,  1.8676e-03, -4.8341e-07, -6.0883e-07,  5.6357e-18,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.3867e-07,
          2.9558e-07, -2.9689e-04,  4.5967e-04, -1.4317e-04, -1.2783e-06,
          8.4117e-07, -3.0959e-07, -2.9695e-04,  4.5967e-04, -1.4317e-04,
          1.6070e-06, -4.1934e-05,  1.4779e-05, -1.5088e-02,  2.2819e-02,
          6.6017e-03, -6.3915e-05,  4.2059e-05, -1.5480e-05, -1.5090e-02,
          2.2822e-02,  6.6030e-03,  8.0352e-05, -1.9162e-01,  8.1881e-01,
         -8.9760e-02,  1.4423e+00, -1.5019e+00,  1.0640e+00,  1.4185e+00,
          1.4308e-01,  1.0860e-03,  9.4490e-01, -1.3915e-01,  4.5646e-01]],
       device='cuda:0')
torques: [ 3.58863138e-10 -2.13966828e-10 -3.99918254e+00 -2.73101646e+00
  2.00000000e+02  8.54795795e-09 -1.07674153e-11  3.21656183e-14
 -3.99918254e+00 -2.73101646e+00  2.00000000e+02 -3.59583103e-10]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.2418,  0.9618,  0.6001,  2.5522, -2.4555,  2.4738,  3.0745,  0.0162,
         -0.3846,  2.8787, -0.2550,  2.2371]], device='cuda:0')
Scaled actions :  tensor([[-0.2418,  0.9618,  0.6001,  2.5522, -2.4555,  2.4738,  3.0745,  0.0162,
         -0.3846,  2.8787, -0.2550,  2.2371]], device='cuda:0')
obs :  tensor([[-7.1312e-02, -3.2956e-02, -1.4088e-02, -1.9709e-03,  2.2886e-03,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -3.7789e-03,
          6.2991e-03, -2.6971e-03,  1.2869e-02, -1.2508e-02,  7.7055e-03,
          1.2606e-02,  1.8246e-03, -1.5728e-03,  1.0108e-02, -6.6576e-03,
          1.3559e-02, -4.0238e-02,  4.4717e-02, -2.3349e-02,  7.5615e-02,
          5.3775e-03, -1.2929e-02,  8.0517e-02,  1.0224e-02, -1.9823e-02,
          6.0103e-02, -7.4904e-02,  1.1637e-03, -2.4178e-01,  9.6179e-01,
          6.0015e-01,  2.5522e+00, -2.4555e+00,  2.4738e+00,  3.0745e+00,
          1.6181e-02, -3.8458e-01,  2.8787e+00, -2.5501e-01,  2.2

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[ 0.2537,  1.0481, -0.7075,  1.6127, -3.5446,  0.5592,  2.8990, -0.2541,
          1.3919,  5.5133,  0.1174, -1.0939]], device='cuda:0')
Scaled actions :  tensor([[ 0.2537,  1.0481, -0.7075,  1.6127, -3.5446,  0.5592,  2.8990, -0.2541,
          1.3919,  5.5133,  0.1174, -1.0939]], device='cuda:0')
obs :  tensor([[-6.7830e-02, -1.9209e-01, -4.0275e-02, -7.2838e-03,  5.6718e-03,
         -9.9996e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -9.9843e-03,
          1.5910e-02,  8.4329e-04,  3.3109e-02, -3.7248e-02,  3.4266e-02,
          3.9786e-02,  2.6039e-03, -6.0220e-03,  3.3612e-02, -1.5098e-02,
          3.5391e-02,  2.0191e-02,  3.5794e-02,  4.4968e-02,  9.3479e-02,
         -1.6392e-01,  1.8426e-01,  1.5183e-01, -8.3035e-04,  4.7908e-03,
          1.0673e-01, -3.6675e-02,  5.7290e-02,  2.5367e-01,  1.0481e+00,
         -7.0747e-01,  1.6127e+00, -3.5446e+00,  5.5921e-01,  2.8990e+00,
         -2.5405e-01,  1.3919e+00,  5.5133e+00,  1.1736e-01, -1.0

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.1888,  1.5526, -0.3097,  0.6368, -3.2281, -1.2033,  1.8288,  0.2664,
          2.1148,  3.2926,  0.0225,  0.7353]], device='cuda:0')
Scaled actions :  tensor([[-0.1888,  1.5526, -0.3097,  0.6368, -3.2281, -1.2033,  1.8288,  0.2664,
          2.1148,  3.2926,  0.0225,  0.7353]], device='cuda:0')
obs :  tensor([[-7.1011e-02, -2.0205e-01, -2.6551e-01, -1.5136e-02,  7.7710e-03,
         -9.9986e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -5.6332e-03,
          2.4467e-02,  4.3110e-03,  4.7550e-02, -6.1052e-02,  4.1307e-02,
          6.8353e-02,  6.2771e-04, -2.9948e-03,  6.7901e-02, -1.5303e-02,
          1.8068e-02,  2.7099e-02,  5.2174e-02, -8.6380e-03,  6.7720e-02,
         -1.6401e-01, -1.4848e-02,  1.6068e-01, -1.3968e-03,  2.5968e-02,
          2.2712e-01, -6.0069e-02, -1.3464e-01, -1.8880e-01,  1.5526e+00,
         -3.0970e-01,  6.3678e-01, -3.2281e+00, -1.2033e+00,  1.8288e+00,
          2.6637e-01,  2.1148e+00,  3.2926e+00,  2.2474e-02,  7.3

In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-0.8247,  1.6465, -0.5390,  0.6378, -3.3666, -0.1695,  1.5300,  0.0944,
          1.3854,  3.0415,  0.0787,  1.0487]], device='cuda:0')
Scaled actions :  tensor([[-0.8247,  1.6465, -0.5390,  0.6378, -3.3666, -0.1695,  1.5300,  0.0944,
          1.3854,  3.0415,  0.0787,  1.0487]], device='cuda:0')
obs :  tensor([[-1.2007e-01, -1.3712e-01, -2.3280e-01, -2.3045e-02,  1.1669e-02,
         -9.9967e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -6.9690e-03,
          3.6823e-02,  2.9732e-03,  5.8192e-02, -7.9888e-02,  2.3856e-02,
          8.5232e-02,  3.0588e-03,  1.4570e-02,  9.3816e-02, -1.0951e-02,
          2.2845e-02,  2.1430e-02,  5.7646e-02, -1.6045e-02,  5.7725e-02,
         -3.1665e-02, -1.5019e-01,  5.1159e-02,  1.0274e-02,  8.0479e-02,
          1.1039e-01,  6.1928e-02,  4.4678e-02, -8.2473e-01,  1.6465e+00,
         -5.3896e-01,  6.3785e-01, -3.3666e+00, -1.6954e-01,  1.5300e+00,
          9.4407e-02,  1.3854e+00,  3.0415e+00,  7.8659e-02,  1.0

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[-0.7987,  1.4311, -0.8858,  1.2630, -3.6506,  0.8512,  1.5939,  0.1063,
          1.0801,  3.4503,  0.4776, -0.6862]], device='cuda:0')
Scaled actions :  tensor([[-0.7987,  1.4311, -0.8858,  1.2630, -3.6506,  0.8512,  1.5939,  0.1063,
          1.0801,  3.4503,  0.4776, -0.6862]], device='cuda:0')
obs :  tensor([[-0.1503, -0.0445, -0.1987, -0.0271,  0.0170, -0.9995,  1.0000,  0.0000,
          0.0000, -0.0196,  0.0520, -0.0057,  0.0788, -0.1271, -0.0063,  0.1042,
          0.0046,  0.0285,  0.1175, -0.0092,  0.0405, -0.0501,  0.0912, -0.1006,
          0.1953, -0.5104, -0.2500,  0.0162,  0.0089,  0.0737,  0.0905, -0.1232,
         -0.0976, -0.7987,  1.4311, -0.8858,  1.2630, -3.6506,  0.8512,  1.5939,
          0.1063,  1.0801,  3.4503,  0.4776, -0.6862]], device='cuda:0')
torques: [-200.          -83.59890036  176.18947707 -118.35922601 -200.
 -200.          200.          200.         -200.          200.
 -200.         -112.57044041]
データ収集: step 7


In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[-0.2828,  1.1193, -0.7955,  1.1417, -2.8778,  1.1072,  1.2243,  0.1663,
          1.2085,  1.6913, -0.1854,  1.8363]], device='cuda:0')
Scaled actions :  tensor([[-0.2828,  1.1193, -0.7955,  1.1417, -2.8778,  1.1072,  1.2243,  0.1663,
          1.2085,  1.6913, -0.1854,  1.8363]], device='cuda:0')
obs :  tensor([[-1.3587e-01, -8.8585e-02, -1.2163e-01, -2.9742e-02,  2.1735e-02,
         -9.9932e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -2.9763e-02,
          6.4542e-02, -1.4658e-02,  1.0186e-01, -1.7042e-01, -1.9261e-03,
          1.1256e-01,  5.9552e-03,  3.6878e-02,  1.4446e-01, -1.2183e-04,
          1.7380e-02, -3.3151e-02,  4.3265e-02, -2.7594e-02,  1.0630e-01,
         -4.9869e-02,  7.2823e-02,  1.6295e-02,  1.1914e-02,  3.7166e-02,
          1.3941e-01, -7.1545e-02, -9.0696e-02, -2.8281e-01,  1.1193e+00,
         -7.9549e-01,  1.1417e+00, -2.8778e+00,  1.1072e+00,  1.2243e+00,
          1.6634e-01,  1.2085e+00,  1.6913e+00, -1.8537e-01,  1.8

In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 0.2953,  1.2380, -1.5259,  2.9450, -2.2446,  0.0047,  1.1199,  0.5955,
          1.6827,  2.3146, -0.6728,  1.2238]], device='cuda:0')
Scaled actions :  tensor([[ 0.2953,  1.2380, -1.5259,  2.9450, -2.2446,  0.0047,  1.1199,  0.5955,
          1.6827,  2.3146, -0.6728,  1.2238]], device='cuda:0')
obs :  tensor([[-0.1401, -0.0453, -0.0796, -0.0329,  0.0278, -0.9991,  1.0000,  0.0000,
          0.0000, -0.0330,  0.0736, -0.0246,  0.1289, -0.1941,  0.0170,  0.1225,
          0.0082,  0.0456,  0.1665, -0.0242,  0.0381, -0.0244,  0.0412, -0.0372,
          0.1147, -0.0189,  0.0439,  0.0149,  0.0174,  0.0272,  0.0677, -0.1757,
          0.0507,  0.2953,  1.2380, -1.5259,  2.9450, -2.2446,  0.0047,  1.1199,
          0.5955,  1.6827,  2.3146, -0.6728,  1.2238]], device='cuda:0')
torques: [ 200.          103.6192642   200.         -200.          200.
 -200.         -100.76467281   64.93100406  200.         -200.
  183.28371829   -2.34524393]
データ収集: step 9


In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[ 0.3352,  1.0936, -1.6561,  3.6331, -1.7530, -0.1185,  0.5742,  0.8403,
          2.4414,  1.0664, -0.7838,  0.0843]], device='cuda:0')
Scaled actions :  tensor([[ 0.3352,  1.0936, -1.6561,  3.6331, -1.7530, -0.1185,  0.5742,  0.8403,
          2.4414,  1.0664, -0.7838,  0.0843]], device='cuda:0')
obs :  tensor([[-0.1162, -0.1511, -0.3291, -0.0363,  0.0319, -0.9988,  1.0000,  0.0000,
          0.0000, -0.0268,  0.0818, -0.0354,  0.1516, -0.2099,  0.0160,  0.1324,
          0.0137,  0.0533,  0.1823, -0.0508,  0.0422,  0.1156,  0.0445, -0.0604,
          0.1023, -0.1990,  0.1212,  0.1233,  0.0503,  0.0403,  0.0718, -0.0695,
         -0.1206,  0.3352,  1.0936, -1.6561,  3.6331, -1.7530, -0.1185,  0.5742,
          0.8403,  2.4414,  1.0664, -0.7838,  0.0843]], device='cuda:0')
torques: [ 200.          133.60592016  200.         -200.          200.
 -145.98034619  200.          101.63906478 -148.28445981  -42.32666366
 -200.          200.        ]
データ収集:

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[ 0.8256,  1.0296, -0.4954,  3.5223, -1.5454,  0.0897,  1.5008,  0.3585,
          0.9488,  0.3618, -2.1824,  2.6373]], device='cuda:0')
Scaled actions :  tensor([[ 0.8256,  1.0296, -0.4954,  3.5223, -1.5454,  0.0897,  1.5008,  0.3585,
          0.9488,  0.3618, -2.1824,  2.6373]], device='cuda:0')
obs :  tensor([[ 6.9999e-02, -2.9908e-01, -4.0197e-01, -4.6335e-02,  3.4030e-02,
         -9.9835e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.5654e-02,
          8.8779e-02, -4.6758e-02,  1.7794e-01, -2.3520e-01,  2.1940e-02,
          1.4021e-01,  1.9888e-02,  6.7972e-02,  1.9348e-01, -7.1376e-02,
          2.9572e-02,  1.2188e-01, -1.7831e-03, -7.8730e-02,  1.3544e-01,
          5.2456e-03, -1.9969e-02,  7.4136e-02, -9.4359e-03,  9.1407e-02,
          4.7710e-02, -7.8522e-02,  1.6675e-02,  8.2565e-01,  1.0296e+00,
         -4.9542e-01,  3.5223e+00, -1.5454e+00,  8.9731e-02,  1.5008e+00,
          3.5852e-01,  9.4878e-01,  3.6179e-01, -2.1824e+00,  2.

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 0.1065,  1.0798, -0.3092,  4.0684, -1.9980,  0.6466,  1.4508,  0.5115,
          0.9952,  1.2602, -1.4895, -0.0968]], device='cuda:0')
Scaled actions :  tensor([[ 0.1065,  1.0798, -0.3092,  4.0684, -1.9980,  0.6466,  1.4508,  0.5115,
          0.9952,  1.2602, -1.4895, -0.0968]], device='cuda:0')
obs :  tensor([[-7.0777e-05, -3.5122e-01, -2.2016e-01, -5.9441e-02,  3.2852e-02,
         -9.9769e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  7.5570e-03,
          9.5120e-02, -6.2232e-02,  2.0484e-01, -2.5560e-01,  2.3338e-02,
          1.5942e-01,  2.3744e-02,  8.9272e-02,  1.9678e-01, -1.0731e-01,
          4.3484e-02,  2.8643e-03,  2.4558e-02, -1.4267e-02,  1.0404e-01,
          6.0736e-04,  7.5032e-02, -4.2553e-02,  2.1863e-02,  1.2787e-01,
          1.2298e-02, -1.5913e-01,  3.8701e-02,  1.0652e-01,  1.0798e+00,
         -3.0923e-01,  4.0684e+00, -1.9980e+00,  6.4660e-01,  1.4508e+00,
          5.1151e-01,  9.9515e-01,  1.2602e+00, -1.4895e+00, -9.

In [26]:
# 既存のforループを置き換え
num_steps = 100
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=6.046, Scaled action max=6.046
Step 1/100, Total steps: 212
steps: 212
actions : tensor([[-7.2925,  5.1942, -1.3544, -4.1728, -9.6965,  1.2872,  2.5646,  0.0591,
          3.6540,  6.0461,  0.5802, -0.9696]], device='cuda:0')
target_dof_pos: tensor([[-1.7978,  1.3607, -1.0181,  0.4946, -3.3203,  0.3846,  0.7147, -0.0360,
          0.1311,  3.2269, -0.7264, -0.2704]], device='cuda:0')
Step 1: Original action max=6.946, Scaled action max=6.946
Step 2: Original action max=6.011, Scaled action max=6.011
Step 21/100, Total steps: 232
steps: 232
actions : tensor([[-3.6007,  2.7221,  0.6244,  6.5506, -1.6507,  3.6144, -2.1567,  1.0438,
          1.6263, -0.8965, -2.1300, -0.2514]], device='cuda:0')
target_dof_pos: tensor([[-0.9563,  0.8412, -0.7334,  3.2519, -1.6820,  0.7547, -0.1788,  0.2522,
         -0.1771,  1.4479, -1.3303,  0.0965]], device='cuda:0')
Step 41/100, Total steps: 252
steps: 252
actions : tensor([[-6.3904e+00,  3.6634e+00,  4.1075e-04, -9.5790e-01

In [66]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [67]:
env.sim.stop()

In [31]:
env.reset()
cnt = 0

In [61]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-terrain2-kp2000kd50-kpkdrand-14_ckpt100_scale1.0_rotorInertia0.1.csv
データ形状: (362, 58)
